# 03 · 训练技巧系统消融 Training Tricks Ablation —— 同一 MLP 上逐项验证

**家族位置**：`01_Fundamentals_MLP` 第 3 个项目（前：`01_Perceptron` → `02_MLP_MNIST`）。本项目完成后，家族学习清单的 8 项全部收口。

**方法**：固定模型（784→256→128→10）、数据（MNIST 20k 子集 + 全量 10k 测试）、轮数（4 epochs）、随机种子（0），**每轮实验只动一个变量**——这就是消融（ablation）的基本功：任何结论都来自受控对照。

**学习目标**
1. 手推一次反向传播（链式法则 + 计算图），并用 autograd 逐参数验证
2. 分清损失函数：CrossEntropy vs MSE、softmax 的 log-sum-exp 数值稳定、label smoothing
3. 亲眼看到每个 trick 值多少分：优化器五件套 / BN·LN·RMSNorm / Dropout·Weight Decay·Label Smoothing·早停
4. 理解激活函数差异与 dead ReLU 现象
5. 万能近似定理：一个隐藏层就能拟合连续函数（sin 回归现场验证）

## 1. 原理速览：反向传播与数值稳定

### 反向传播 = 计算图上的链式法则

前向把输入一路算到损失 $L$；反向则沿**同一条计算图逆行**，把 $\partial L/\partial L = 1$ 一路乘开：

$$\frac{\partial L}{\partial W_1} = \underbrace{\frac{\partial L}{\partial z_2}}_{\text{softmax+CE}} \cdot \underbrace{\frac{\partial z_2}{\partial a_1}}_{W_2^\top} \cdot \underbrace{\frac{\partial a_1}{\partial z_1}}_{\text{ReLU}'(z_1)} \cdot \underbrace{\frac{\partial z_1}{\partial W_1}}_{X}$$

两个经典结论会现场验证：
- **softmax + CrossEntropy 的联合梯度恰好是 $p - y_{\text{onehot}}$**（所以框架把两者合并成一个算子）
- **ReLU 的局部梯度是 0/1 指示矩阵**（负区梯度为零 → dead ReLU 的根源）

### softmax 为什么会溢出

$e^{1000}$ 直接超出 float32 上限 → `inf` → `nan`。工程解法是 **log-sum-exp 技巧**：$\log p_y = z_y - \log\sum_k e^{z_k}$，先减去最大值再指数。`F.cross_entropy` 内置了它——这就是"永远用现成 CE，别手写 softmax"的原因。

In [ ]:
import sys
from functools import partial
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import load_mnist_torch
from common.engine import fit
from common.models import MLP
from common.utils import activation_zero_fractions, set_seed, setup_chinese_font

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("设备:", DEVICE, "| torch:", torch.__version__)

## 2. 实验数据

消融实验用 MNIST **20k 训练子集**（单次实验约 3 秒，15+ 组对照几分钟跑完），评估用**全量 10k 测试集**保证口径一致。固定 `set_seed(0)`，所有数字可复现。

In [ ]:
Xtr, ytr, Xte, yte = load_mnist_torch(str(ROOT / "data"), flatten=True)
SUBSET, EPOCHS = 20000, 4
tr = DataLoader(TensorDataset(Xtr[:SUBSET], ytr[:SUBSET]), batch_size=128, shuffle=True)
te = DataLoader(TensorDataset(Xte, yte), batch_size=512)
print(f"训练子集: {SUBSET} | 测试全量: {len(yte)} | 每组 {EPOCHS} epochs")

## 3. Part A · 手推反向传播，用 autograd 逐参数验证

一个 2→5→2 的小网络，NumPy 手写前向 + 反向（核心三步：softmax+CE 联合梯度、$W_2^\top$ 回传、ReLU 指示矩阵），再与 PyTorch autograd 对比每一个梯度张量。**差异 < 1e-6 才算通过**——这是"链式法则我推对了"的硬证据。

In [ ]:
set_seed(0)
N, D, H, C = 8, 2, 5, 2
X = torch.randn(N, D)
W1 = torch.randn(D, H) * 0.5; b1 = torch.zeros(H)
W2 = torch.randn(H, C) * 0.5; b2 = torch.zeros(C)
y = torch.tensor([0, 1, 0, 1, 1, 0, 1, 0])

# ---- PyTorch autograd 基准 ----
Xa, W1a, b1a, W2a, b2a = (t.clone().requires_grad_(True) for t in (X, W1, b1, W2, b2))
scores = torch.relu(Xa @ W1a + b1a) @ W2a + b2a
loss_auto = F.cross_entropy(scores, y)
loss_auto.backward()

# ---- NumPy 手推 ----
Xn, W1n, b1n, W2n, b2n = (t.detach().numpy() for t in (X, W1, b1, W2, b2))
onehot = F.one_hot(y, C).float().numpy()
z1 = Xn @ W1n + b1n                       # 前向 1：线性
a1 = np.maximum(z1, 0.0)                  # 前向 2：ReLU
s = a1 @ W2n + b2n                        # 前向 3：logits
exp = np.exp(s - s.max(1, keepdims=True)) # 减最大值防溢出
p = exp / exp.sum(1, keepdims=True)
L = float(-np.log(p[np.arange(N), y.numpy()]).mean())

dz2 = (p - onehot) / N                    # softmax+CE 联合梯度 = p - y
gW2 = a1.T @ dz2; gb2 = dz2.sum(0)        # 输出层参数梯度


In [ ]:
da1 = dz2 @ W2n.T
dz1 = da1 * (z1 > 0.0)                    # ReLU 局部梯度：0/1 指示矩阵
gW1 = Xn.T @ dz1; gb1 = dz1.sum(0)

print(f"loss: autograd={loss_auto.item():.6f} | 手推={L:.6f}")
for name, mine, ref in [("dW1", gW1, W1a.grad.numpy()), ("db1", gb1, b1a.grad.numpy()),
                        ("dW2", gW2, W2a.grad.numpy()), ("db2", gb2, b2a.grad.numpy())]:
    diff = float(np.abs(mine - ref).max())
    print(f"{name}: max|手推 - autograd| = {diff:.2e}")
    assert diff < 1e-6, f"{name} 梯度不一致"
print("✔ 手推反向传播与 autograd 完全一致")

## 4. Part B · 损失函数：溢出现场 + CE vs MSE 同台

`CrossEntropyLoss` 的内部是 log-sum-exp，天然数值稳定；手写 softmax 会溢出。另外，"分类为什么不用 MSE"不用背结论——同一模型同一数据跑一遍就知道。

In [ ]:
# 1) 溢出现场：logits 放大 1000 倍
big = torch.randn(4, 10) * 1000
naive = big.exp() / big.exp().sum(1, keepdim=True)
print("朴素 softmax 出现 nan:", bool(torch.isnan(naive).any()))
print("F.cross_entropy（内置 log-sum-exp）:", F.cross_entropy(big, torch.tensor([1, 0, 3, 2])).item(), "← 数值稳定")

# 2) 同一模型同一数据：CE vs MSE（MSE 打在 softmax 概率上）
def mse_on_probs(logits, yb):
    return F.mse_loss(logits.softmax(1), F.one_hot(yb, 10).float())

for name, crit in [("CrossEntropy", nn.CrossEntropyLoss()), ("MSE(softmax 概率)", mse_on_probs)]:
    set_seed(0)
    h = fit(MLP(784, (256, 128), 10, dropout=0.2), tr, te, epochs=EPOCHS, lr=1e-3,
            verbose=False, criterion=crit)
    print(f"{name:18s} val_acc={h['val_acc'][-1]:.2%} | val_loss={h['val_loss'][-1]:.4f}")

## 5. Part B2 · 万能近似定理：一个隐藏层拟合 sin

万能近似定理（Universal Approximation Theorem）：**单隐层 + 足够宽 + 非线性激活**，理论上能以任意精度逼近任意连续函数。注意它只保证"存在这样的网络"，不保证 SGD 找得到、也不保证宽度的经济性——这正是后来走向"深"的动机。现场用 1→64→1 的网络回归 $\sin(2x)$。

In [ ]:
set_seed(0)
rng = np.random.default_rng(0)
x = np.linspace(-np.pi, np.pi, 512).astype(np.float32)
yfit = np.sin(2 * x)
xt = torch.tensor(x).unsqueeze(1)
yt = torch.tensor(yfit).unsqueeze(1)

uat = MLP(1, (64,), 1)
opt = torch.optim.Adam(uat.parameters(), lr=1e-2)
for step in range(1, 3001):
    opt.zero_grad()
    loss = F.mse_loss(uat(xt), yt)
    loss.backward()
    opt.step()
    if step % 1000 == 0:
        print(f"step {step:4d} | mse = {loss.item():.5f}")

plt.figure(figsize=(7, 3.2))
plt.plot(x, yfit, "k--", lw=1.5, label="sin(2x)")
plt.plot(x, uat(xt).detach().numpy(), color="#DD8452", lw=2, label="MLP(1→64→1) 拟合")
plt.legend(fontsize=9); plt.title("万能近似定理现场：一个隐藏层拟合连续函数")
plt.tight_layout()
plt.savefig(FIGS / "fig1_uat.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"最终拟合 mse = {loss.item():.5f} → assert < 0.01")
assert loss.item() < 0.01

## 6. Part C · 核心消融：四大组，一次只动一个变量

| 组 | 变量 | 固定项 |
|---|---|---|
| ① 优化器 | SGD / +Momentum / RMSProp / AdamW（含 lr 对照） | 其余全同 |
| ② 归一化 | 无 / BatchNorm / LayerNorm / RMSNorm | 去掉 Dropout，隔离变量 |
| ③ 正则化 | 无 / Dropout / Weight Decay / Label Smoothing（+早停单测） | 其余全同 |

解读要点：不是"哪个最强"，而是**每个组件在自己的轴上贡献了多少**——以及 lr 对 SGD 系的决定性影响。

In [ ]:
rows = []

def add(group, name, hist, note=""):
    rows.append({"group": group, "name": name, "acc": hist["val_acc"][-1],
                 "loss": hist["val_loss"][-1], "note": note})

def train(model=None, **kw):
    set_seed(0)
    if model is None:
        model = MLP(784, (256, 128), 10, dropout=0.2)
    return fit(model, tr, te, epochs=EPOCHS, lr=kw.pop("lr", 1e-3), verbose=False, **kw)

add("基线", "Adam(1e-3)+Dropout0.2", train())

for name, cls, lr in [
    ("SGD(1e-3)", torch.optim.SGD, 1e-3),
    ("SGD+Momentum(1e-3)", partial(torch.optim.SGD, momentum=0.9), 1e-3),
    ("RMSProp(1e-3)", torch.optim.RMSprop, 1e-3),
    ("AdamW(1e-3)", torch.optim.AdamW, 1e-3),
    ("SGD+Momentum(0.1)", partial(torch.optim.SGD, momentum=0.9), 1e-1),
]:
    add("①优化器", name, train(optimizer_cls=cls, lr=lr))

for name, norm in [("无", None), ("BatchNorm", "bn"), ("LayerNorm", "ln"), ("RMSNorm", "rms")]:
    add("②归一化", name, train(model=MLP(784, (256, 128), 10, norm=norm)))

add("③正则化", "无正则化", train(model=MLP(784, (256, 128), 10)))
add("③正则化", "Dropout 0.2", train())
add("③正则化", "AdamW+wd=1e-2", train(model=MLP(784, (256, 128), 10),
                                      optimizer_cls=torch.optim.AdamW, weight_decay=1e-2))
add("③正则化", "LabelSmoothing 0.1", train(model=MLP(784, (256, 128), 10),
                                            criterion=nn.CrossEntropyLoss(label_smoothing=0.1)))

print(f"{'组':6s} | {'配置':24s} | {'val_acc':>7s} | val_loss")
print("-" * 60)
for r in rows:
    print(f"{r['group']:6s} | {r['name']:24s} | {r['acc']:7.2%} | {r['loss']:.4f}")

In [ ]:
colors = {"基线": "#55A868", "①优化器": "#4C72B0", "②归一化": "#DD8452", "③正则化": "#C44E52"}
fig, ax = plt.subplots(figsize=(9, 5.2))
ys = np.arange(len(rows))
ax.barh(ys, [r["acc"] for r in rows], color=[colors[r["group"]] for r in rows])
ax.set_yticks(ys); ax.set_yticklabels([r["name"] for r in rows], fontsize=9)
ax.invert_yaxis(); ax.set_xlim(0.5, 1.0)
ax.set_xlabel("最终 val_acc（MNIST 20k 子集 · 4 epochs · seed=0）")
ax.set_title("训练技巧消融：同一 MLP、同一数据，每次只动一个变量")
for yy, r in zip(ys, rows):
    ax.text(r["acc"] + 0.004, yy, f"{r['acc']:.2%}", va="center", fontsize=8)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in colors.values()]
ax.legend(handles, colors.keys(), loc="lower right", fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig2_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

### 早停（Early Stopping）：给训练加"刹车"

监控 val_acc，连续 `patience` 轮无提升就停——省算力，也天然防过拟合。下面用**无 Dropout** 的配置（更容易平台期）演示：计划 10 轮，看它第几轮刹住。

In [ ]:
set_seed(0)
h_es = fit(MLP(784, (256, 128), 10), tr, te, epochs=10, lr=1e-3, es_patience=2, verbose=False)
stop_ep = len(h_es["val_acc"])
print(f"早停（patience=2）：计划 10 轮 → 实际第 {stop_ep} 轮停止 | 最佳 val_acc={max(h_es['val_acc']):.2%}")
print("逐轮 val_acc:", [f"{a:.2%}" for a in h_es["val_acc"]])
assert stop_ep < 10, "早停未触发"
print(f"✔ 早停生效：省下 {10 - stop_ep} 轮训练")

## 7. Part D · 激活函数五件套 + dead ReLU 现场取证

同网络同数据，只换激活函数：Sigmoid / Tanh / ReLU / GELU / SiLU。同时用 hook 统计**训练后激活输出恰好为 0 的比例**——ReLU 系独有的"死亡"现象（负区梯度恒为 0，神经元再也学不动）；Sigmoid/Tanh 还要额外面对两端饱和（梯度≈0，学得慢）。

In [ ]:
act_rows = []
for name, act in [("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU),
                  ("GELU", nn.GELU), ("SiLU", nn.SiLU)]:
    set_seed(0)
    model = MLP(784, (256, 128), 10, dropout=0.0, activation=act)
    h = fit(model, tr, te, epochs=EPOCHS, lr=1e-3, verbose=False)
    zero = float(np.mean(activation_zero_fractions(model, Xtr[:512])))
    act_rows.append({"name": name, "acc": h["val_acc"][-1], "zero": zero})
    print(f"{name:8s} val_acc={h['val_acc'][-1]:.2%} | 激活输出=0 的比例: {zero:.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
names = [r["name"] for r in act_rows]
axes[0].bar(names, [r["acc"] for r in act_rows], color="#4C72B0")
axes[0].set_title("最终 val_acc"); axes[0].set_ylim(0.5, 1.0)
axes[1].bar(names, [r["zero"] for r in act_rows], color="#C44E52")
axes[1].set_title("激活输出恰为 0 的比例（训练后）")
for ax in axes:
    ax.set_xlabel("激活函数")
    for i, r in enumerate(act_rows):
        v = r["acc"] if ax is axes[0] else r["zero"]
        ax.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
plt.tight_layout()
plt.savefig(FIGS / "fig3_activation.png", dpi=150, bbox_inches="tight")
plt.show()
relu_row = next(r for r in act_rows if r["name"] == "ReLU")
assert relu_row["zero"] > 0, "ReLU 应有非零死亡比例"
print("✔ dead ReLU 现场取证完成")

## 8. Part E · 归一化的深层差异：为什么 Transformer 用 LN 而不是 BN

BN 沿 **batch 维**统计均值方差 → 依赖 batch 内其他样本、训练/推理行为不一致、对 batch size 敏感。LN/RMSNorm 沿**特征维**归一化 → 与 batch 无关。现场对照：把 batch 压到 2，BN 应明显受伤，LN 几乎无感——这正是序列模型（变长、batch 依赖弱）选 LN 的实践原因。

In [ ]:
demo = []
for bs in (128, 2):
    set_seed(0)
    tr_small = DataLoader(TensorDataset(Xtr[:5000], ytr[:5000]), batch_size=bs, shuffle=True)
    for norm, label in [("bn", "BatchNorm"), ("ln", "LayerNorm")]:
        h = fit(MLP(784, (256, 128), 10, norm=norm), tr_small, te, epochs=2, lr=1e-3, verbose=False)
        demo.append({"label": label, "bs": bs, "acc": h["val_acc"][-1]})
        print(f"batch_size={bs:3d} | {label}: val_acc={h['val_acc'][-1]:.2%}")

fig, ax = plt.subplots(figsize=(6, 3.5))
w = 0.35
xs = np.arange(2)
for i, label in enumerate(["BatchNorm", "LayerNorm"]):
    vals = [d["acc"] for d in demo if d["label"] == label]
    bars = ax.bar(xs + (i - 0.5) * w, vals, w, label=label,
                  color="#DD8452" if i == 0 else "#4C72B0")
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1%}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(xs); ax.set_xticklabels(["batch=128", "batch=2"])
ax.set_ylim(0.5, 1.0); ax.set_ylabel("val_acc")
ax.set_title("把 batch 压到 2：BN 受伤，LN 无感（5000 子集 · 2 epochs）")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig4_bn_batch.png", dpi=150, bbox_inches="tight")
plt.show()
bn = {d["bs"]: d["acc"] for d in demo if d["label"] == "BatchNorm"}
assert bn[2] < bn[128], "BN 应在小 batch 下退化"
print("✔ BN 的 batch 依赖实证完成")

## 9. 总结：清单收口

| 家族清单项 | 在哪学的 | 关键结论 |
|---|---|---|
| Perceptron | 01 | 线性天花板（XOR 卡 ~56%/47%） |
| MLP + 万能近似 | 02 / 03 PartB2 | 隐藏层 + 非线性 = 逼近任意连续函数 |
| 反向传播 | 03 PartA | softmax+CE 联合梯度 = p−y；ReLU' = 0/1；autograd 逐项验证 |
| 激活函数五件套 | 03 PartD | ReLU 系快；Sigmoid 慢在饱和；dead ReLU 可测可证 |
| 归一化 BN/LN/RMSNorm | 03 PartC② + PartE | BN 依赖 batch（小 batch 受伤），LN 与 batch 无关 → Transformer 选 LN |
| 优化器五件套 | 03 PartC① | 自适应方法对 lr 不敏感；SGD 系靠 Momentum + 大 lr |
| 正则化四件套 | 02 + 03 PartC③ | Dropout/WD/LabelSmoothing/早停各有其位，过量会反噬 |
| 损失函数 CE/MSE | 03 PartB | 分类用 CE；softmax 溢出 → log-sum-exp |

**三条元结论**
1. **控制变量才能归因**：所有"trick 有效"的结论都来自同模型同数据只动一个变量的对照
2. **超参之间有耦合**：SGD 的差距一半来自 Momentum，一半来自 lr——报告结论必须写清配置
3. **理论 → 现象 → 证据**：dead ReLU 不是 folklore，本项目的 hook 统计给了它一个数字

**下一步**：`02_CNN_Family/01_LeNet_MNIST`——给网络装上"看图"的归纳偏置，与 MLP 的 98.14% 正面对比。